# Terrain analysis from a DEM

Derive terrain products from an elevation raster:

- **`slope`** — steepness (degrees or percent).
- **`aspect`** — the compass direction a slope faces.
- **`hillshade`** — shaded relief for a given sun position.
- **`proximity`** — distance from each cell to the nearest target cell.

`slope` / `aspect` / `hillshade` return NumPy arrays; `proximity` returns a `Dataset`.

## Setup

In [1]:
import os

os.environ['MPLBACKEND'] = 'Agg'  # force a headless backend before any import

import matplotlib

matplotlib.use('Agg', force=True)

import shutil
import tempfile
from pathlib import Path

import numpy as np


DATA = Path('../../../examples/data')
if not DATA.exists():
    DATA = Path('examples/data')
WORK = Path(tempfile.mkdtemp(prefix='pyramids-t2-'))
DATA.is_dir(), WORK.is_dir()

(True, True)

In [2]:
from pyramids.dataset import Dataset

dem = Dataset.read_file(str(DATA / 'dem' / 'DEM5km_Rhine_burned_acc.tif'))
dem.shape, dem.epsg, dem.cell_size

2026-06-08 23:51:01 | INFO | pyramids.base.config | Logging is configured.


((1, 125, 93), 4647, 5000.0)

## Slope and aspect

`units='degrees'` (default) or `'percent'` for slope; aspect is in degrees.

In [3]:
slope = dem.slope(units='degrees')
aspect = dem.aspect()
slope.shape, float(np.nanmin(slope)), float(np.nanmax(slope)), aspect.shape

((125, 93), 0.0, 89.99988068150554, (125, 93))

## Hillshade

Shaded relief for a sun at `azimuth` (compass °) and `altitude` (° above horizon).

In [4]:
shade = dem.hillshade(azimuth=315.0, altitude=45.0)
shade.shape, float(np.nanmin(shade)), float(np.nanmax(shade))

((125, 93), 0.0, 245.09083049498994)

## Proximity

Euclidean distance from every cell to the nearest cell holding a target value (here, cells
equal to 1). Returns a `Dataset` whose values are distances in the raster's units (`GEO`).

In [5]:
dist = dem.proximity(target_values=[1], distance_units='GEO')
type(dist).__name__, dist.shape

('Dataset', (1, 125, 93))

## Notes

- All of these also accept `chunks=` to run lazily on a Dask array — see the
  [Dask quickstart — Dataset](../dask/dataset.ipynb).
- To render a hillshade or slope, wrap the array back into a `Dataset` and use
  [Visualization](visualization.ipynb).